In [8]:
"""
LIF + OU 突触电流神经元模型: 给定目标 ISI 均值 m 与方差 sigma^2,
反解参数 (g_syn, mu, sigma), 并用直接数值模拟(2维 SDE)验证。

模型:
    dV/dt      = -g_L V + g_syn I_syn
    tau dI_syn = (-I_syn + mu) dt + sigma dW

理论基础 (白噪声极限 tau -> 0):
    g_syn*I_syn(t) 的自协方差积分 = g_syn^2 * sigma^2 (与 tau 无关),
    故在该极限下系统退化为等效 1 维 LIF-OU 模型(白噪声驱动):
        dV = (-g_L V + g_syn*mu) dt + g_syn*sigma dW(t)
    通过时间重标度 t' = g_L t 化为单位泄漏形式:
        dV = (mu_eff - V) dt' + sqrt(2 D_eff) dW'(t')
        mu_eff = g_syn*mu / g_L,   D_eff = g_syn^2 sigma^2 / (2 g_L)
    该化简后系统的首次穿越时刻矩 (m1, m2) 用 Ricciardi 双重积分公式精确计算,
    再换算回原时间单位: E[T] = m1/g_L + tau_ref,  Var(T) = (m2-m1^2)/g_L^2
"""

import numpy as np
from scipy import integrate, optimize
from scipy.stats import norm
from scipy.interpolate import interp1d


# ---------------------------------------------------------------------
# 1. 等效白噪声 LIF 的 FPT 一阶、二阶矩 (单位泄漏形式, Ricciardi 公式)
# ---------------------------------------------------------------------
def _psi(v, mu_eff, D_eff):
    return np.exp(-(v - mu_eff) ** 2 / (2 * D_eff))


def _m1(v, mu_eff, D_eff, Vth):
    """m1(v) = E_v[T'],  单位泄漏 OU 过程的首次穿越时间均值"""
    def inner(y):
        return np.sqrt(2 * np.pi * D_eff) * norm.cdf((y - mu_eff) / np.sqrt(D_eff))

    def outer(y):
        return inner(y) / _psi(y, mu_eff, D_eff)

    val, _ = integrate.quad(outer, v, Vth, limit=200)
    return val / D_eff


def _m2(v, mu_eff, D_eff, Vth, n_grid=120):
    """m2(v) = E_v[T'^2], 通过递推积分公式计算 (需先在网格上插值 m1)"""
    lo = mu_eff - 12 * np.sqrt(D_eff)
    grid = np.linspace(min(v, lo) - 1.0, Vth, n_grid)
    m1_vals = np.array([_m1(g, mu_eff, D_eff, Vth) for g in grid])
    m1_interp = interp1d(grid, m1_vals, kind="cubic", fill_value="extrapolate")

    def inner(y):
        integrand = lambda z: _psi(z, mu_eff, D_eff) * m1_interp(z)
        val, _ = integrate.quad(integrand, lo, y, limit=200)
        return val

    def outer(y):
        return inner(y) / _psi(y, mu_eff, D_eff)

    val, _ = integrate.quad(outer, v, Vth, limit=200)
    return 2 * val / D_eff


def T_moments(mu_eff, D_eff, Vth, Vr, gL, tau_ref):
    """返回原始时间单位下 ISI 的 (均值, 方差)"""
    m1p = _m1(Vr, mu_eff, D_eff, Vth)
    m2p = _m2(Vr, mu_eff, D_eff, Vth)
    var_p = m2p - m1p ** 2
    ET = m1p / gL + tau_ref
    VarT = var_p / gL ** 2
    return ET, VarT


# ---------------------------------------------------------------------
# 2. 参数反解: 给定目标 (m, sigma_T^2), 求 (mu_eff, D_eff), 再映射回
#    (g_syn, mu, sigma) -- sigma 取为自由参数 (3 参数 2 方程,有 1 维自由度)
# ---------------------------------------------------------------------
def solve_effective_params(target_mean, target_var, Vth, Vr, gL, tau_ref,
                            init=(2.0, 0.5)):
    def eqs(x):
        mu_eff, log_D = x
        D_eff = np.exp(log_D)
        ET, VarT = T_moments(mu_eff, D_eff, Vth, Vr, gL, tau_ref)
        return [ET - target_mean, VarT - target_var]

    x0 = [init[0], np.log(init[1])]
    sol = optimize.fsolve(eqs, x0, full_output=True)
    x, info, ier, msg = sol
    if ier != 1:
        raise RuntimeError(f"求解未收敛: {msg}")
    return x[0], np.exp(x[1])


def map_to_physical_params(mu_eff, D_eff, gL, sigma):
    """给定自由选取的 sigma, 反解 g_syn 与 mu"""
    gsyn = np.sqrt(2 * gL * D_eff) / sigma
    mu = mu_eff * gL / gsyn
    return gsyn, mu


def construct_parameters(target_mean, target_var, gL, Vth, Vr, tau_ref,
                          sigma_choice=1.0):
    """主接口: 给定目标 m, sigma_T^2 及固定常数, 返回 (g_syn, mu, sigma)"""
    mu_eff, D_eff = solve_effective_params(target_mean, target_var,
                                            Vth, Vr, gL, tau_ref)
    gsyn, mu = map_to_physical_params(mu_eff, D_eff, gL, sigma_choice)
    return gsyn, mu, sigma_choice, (mu_eff, D_eff)


# ---------------------------------------------------------------------
# 3. 直接数值模拟原始 2 维 SDE 系统 (Euler-Maruyama), 用于验证
# ---------------------------------------------------------------------
def simulate_isi(gL, gsyn, mu, sigma, tau, Vth, Vr, tau_ref,
                  n_trials=2000, dt=2e-4, t_max=40, seed=0):
    rng = np.random.default_rng(seed)
    ISIs = np.full(n_trials, np.nan)
    sqdt = np.sqrt(dt)
    I_std = np.sqrt(sigma ** 2 / (2 * tau))  # I_syn 平稳标准差

    for k in range(n_trials):
        V = Vr
        Isyn = mu + I_std * rng.standard_normal()  # 从平稳分布中抽取复位时刻电流
        t = 0.0
        while t < t_max:
            V += (-gL * V + gsyn * Isyn) * dt
            Isyn += (-Isyn + mu) / tau * dt + (sigma / tau) * sqdt * rng.standard_normal()
            t += dt
            if V >= Vth:
                ISIs[k] = t + tau_ref
                break
    return ISIs


# ---------------------------------------------------------------------
# 4. 主程序: 演示构造参数并验证
# ---------------------------------------------------------------------
if __name__ == "__main__":
    # ---- 固定常数 ----
    gL = 1.0
    Vth = 1.0
    Vr = 0.0
    tau_ref = 0.0

    # ---- 目标 ISI 统计量 ----
    target_mean = 0.6
    target_var = 0.08

    print(f"目标: E[T] = {target_mean}, Var(T) = {target_var}\n")

    gsyn, mu, sigma, (mu_eff, D_eff) = construct_parameters(
        target_mean, target_var, gL, Vth, Vr, tau_ref, sigma_choice=1.0
    )
    print(f"构造参数: g_syn = {gsyn:.5f}, mu = {mu:.5f}, sigma = {sigma:.5f}")
    print(f"(等效参数: mu_eff = {mu_eff:.5f}, D_eff = {D_eff:.5f})\n")

    # ---- 理论自洽性检查 (白噪声极限公式本身) ----
    ET_theory, VarT_theory = T_moments(mu_eff, D_eff, Vth, Vr, gL, tau_ref)
    print(f"理论自洽性检查: E[T] = {ET_theory:.5f}, Var(T) = {VarT_theory:.5f}")
    print("(应与目标值几乎完全相等)\n")

    # ---- 用原始 2 维 SDE 系统做数值模拟, 在不同 tau 下验证白噪声极限近似 ----
    print(f"{'tau':>8} | {'sim E[T]':>10} | {'sim Var(T)':>11} | {'mean_err%':>10} | {'var_err%':>9} | n_trials")
    print("-" * 75)
    tau_list = [0.0005, 0.001, 0.005, 0.01, 0.02, 0.05]
    for tau_val in tau_list:
        dt = min(2e-4, tau_val / 15)
        ISIs = simulate_isi(gL, gsyn, mu, sigma, tau_val, Vth, Vr, tau_ref,
                             n_trials=6000, dt=dt, t_max=25, seed=123)
        ISIs = ISIs[~np.isnan(ISIs)]
        mean_err = 100 * abs(ISIs.mean() - target_mean) / target_mean
        var_err = 100 * abs(ISIs.var() - target_var) / target_var
        print(f"{tau_val:8.4f} | {ISIs.mean():10.4f} | {ISIs.var():11.4f} | "
              f"{mean_err:10.2f} | {var_err:9.2f} | {len(ISIs)}")

    print("\n结论: tau 越小, 模拟结果越接近理论目标值 (均值/方差相对误差单调下降),")
    print("验证了白噪声极限 (时间尺度分离假设 tau << E[T]) 下参数构造方法的有效性。")


目标: E[T] = 0.6, Var(T) = 0.08

构造参数: g_syn = 0.62393, mu = 3.36736, sigma = 1.00000
(等效参数: mu_eff = 2.10101, D_eff = 0.19465)

理论自洽性检查: E[T] = 0.60000, Var(T) = 0.08000
(应与目标值几乎完全相等)

     tau |   sim E[T] |  sim Var(T) |  mean_err% |  var_err% | n_trials
---------------------------------------------------------------------------
  0.0005 |     0.6036 |      0.0803 |       0.60 |      0.33 | 6000
  0.0010 |     0.6122 |      0.0843 |       2.04 |      5.31 | 6000
  0.0050 |     0.6313 |      0.0870 |       5.22 |      8.79 | 6000
  0.0100 |     0.6426 |      0.0916 |       7.10 |     14.48 | 6000
  0.0200 |     0.6591 |      0.0965 |       9.86 |     20.60 | 6000
  0.0500 |     0.6834 |      0.1016 |      13.90 |     27.05 | 6000

结论: tau 越小, 模拟结果越接近理论目标值 (均值/方差相对误差单调下降),
验证了白噪声极限 (时间尺度分离假设 tau << E[T]) 下参数构造方法的有效性。
